In [1]:
import pandas as pd
intl = pd.read_csv('data/international_results.csv')
print(intl.shape)
print(intl.head())

(46442, 9)
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1  1873-03-08   England  Scotland           4           2   Friendly   London   
2  1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3  1875-03-06   England  Scotland           2           2   Friendly   London   
4  1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


### Convert date column and check data quality

Right now the date column is just text (a string), even though it looks like a date. We need to convert it to an actual datetime type so pandas can sort and compare dates properly — this is critical since our whole approach depends on processing matches in chronological order.

In [2]:
intl['date'] = pd.to_datetime(intl['date'])
print(intl.dtypes)

date          datetime64[us]
home_team                str
away_team                str
home_score             int64
away_score             int64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object


In [3]:
print(intl.isnull().sum())
#print()
print(intl['neutral'].dtype)
print(intl['neutral'].unique())

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64
bool
[False  True]


### Filter to matches from 2000 onwards

Here's the reasoning before you write the code: we have data going back to 1872, but we don't want to train on 150 years of football. The game has changed completely — tactics, athleticism, squad management are all unrecognisable from 50 years ago. More importantly, Elo ratings need time to "warm up" — if we start from 2012, every team begins at the same rating (1500), so Brazil and Bhutan look identical on day 1. Starting from 2000 gives ratings ~12 years to diverge into realistic values before we start training

In [4]:
df = intl[intl['date'] >= '2000-01-01']
df = df.sort_values('date').reset_index(drop=True) #if you don't have the reset_index function, the dataframe(df)
# will start from the index of corresponding intl columns. For example it may start from 1534 and go on 1544, 1545
# and so on but it is better to start indexing the columns from 0 onwards. drop = true performs the function of 
# dropping the old indices of the corresponding intl columns
print(df.shape)
print(df['date'].min(), df['date'].max())
print(df.head(3))

(22795, 9)
2000-01-04 00:00:00 2024-03-26 00:00:00
        date            home_team away_team  home_score  away_score  \
0 2000-01-04                Egypt      Togo           2           1   
1 2000-01-07              Tunisia      Togo           7           0   
2 2000-01-08  Trinidad and Tobago    Canada           0           0   

  tournament           city              country  neutral  
0   Friendly          Aswan                Egypt    False  
1   Friendly          Tunis              Tunisia    False  
2   Friendly  Port of Spain  Trinidad and Tobago    False  


In [5]:
from collections import deque, defaultdict

IINITIAL_ELO = 1500
K = 30

def expected_score(rating_a, rating_b):
    return 1/(1+10**((rating_b-rating_a)/400))

def actual_score(goals_for, goals_against):
    if goals_for>goals_against:
        return 1.0
    if goals_for == goals_against:
        return 0.5
    else:
        return 0.0
    
def goal_diff_multiplier(goal_diff):
    gd = abs(goal_diff)
    if gd<=1:
        return 1
    elif gd == 2:
        return 1.5
    else:
        return (11+gd)/8

elo = {}
home_elo_pre = []
away_elo_pre = []

for idx, row in df.iterrows():
    home, away = row['home_team'], row['away_team']
    hs, as_ = row['home_score'], row['away_score']

    r_home = elo.get(home, IINITIAL_ELO)
    r_away = elo.get(away, IINITIAL_ELO)

    home_elo_pre.append(r_home)
    away_elo_pre.append(r_away)

    exp_home = expected_score(r_home, r_away)
    exp_away = expected_score(r_away, r_home)

    act_home = actual_score(hs, as_)
    act_away = actual_score(as_, hs)

    gd_mult = goal_diff_multiplier(hs-as_)

    elo[home] = r_home + K*gd_mult*(act_home-exp_home)
    elo[away] = r_away + K*gd_mult*(act_away-exp_away)

df['home_elo'] = home_elo_pre
df['away_elo'] = away_elo_pre

print(df[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'home_elo', 'away_elo']].head(10))


        date            home_team away_team  home_score  away_score  home_elo  \
0 2000-01-04                Egypt      Togo           2           1    1500.0   
1 2000-01-07              Tunisia      Togo           7           0    1500.0   
2 2000-01-08  Trinidad and Tobago    Canada           0           0    1500.0   
3 2000-01-09         Burkina Faso     Gabon           1           1    1500.0   
4 2000-01-09            Guatemala   Armenia           1           1    1500.0   
5 2000-01-09          Ivory Coast     Egypt           2           0    1500.0   
6 2000-01-09               Mexico      Iran           2           1    1500.0   
7 2000-01-11              Bermuda    Canada           0           2    1500.0   
8 2000-01-11         Burkina Faso  Cameroon           2           2    1500.0   
9 2000-01-13              Senegal  Cameroon           0           0    1500.0   

   away_elo  
0    1500.0  
1    1485.0  
2    1500.0  
3    1500.0  
4    1500.0  
5    1515.0  
6    1500.

In [6]:
top_elos = pd.Series(elo).sort_values(ascending=False).head(10)
print(top_elos)

Argentina      2114.612327
France         2032.747326
Brazil         2030.529149
Spain          2014.474544
England        2012.808228
Colombia       2009.167216
Portugal       1995.020067
Belgium        1988.424290
Netherlands    1976.626190
Uruguay        1972.883102
dtype: float64
